# 5.3 Lab: Validação Cruzada e Bootstrap

Neste laboratório, o foco é aplicar técnicas de reamostragem para estimar erro de teste e incerteza de estimadores. A sequência segue três blocos: abordagem por conjunto de validação, validação cruzada e bootstrap.

Alguns comandos podem demorar mais, principalmente quando usamos LOOCV, porque o modelo é reestimado muitas vezes.

Começamos carregando os pacotes principais usados ao longo do laboratório.

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from sklearn.model_selection import train_test_split

Agora importamos funções adicionais necessárias para validação cruzada, bootstrap e integração entre `statsmodels` e `sklearn`.

In [2]:
from functools import partial
from sklearn.model_selection import      (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

## 5.3.1 Abordagem do conjunto de validação

A abordagem de validação separa os dados em duas partes. Uma parte é usada para ajustar o modelo, e a outra parte é usada para estimar o erro em dados não vistos. Aqui usamos o conjunto `Auto`, com 392 observações, e separamos metade para treino e metade para validação.

O argumento `random_state=0` fixa a aleatoriedade, permitindo reproduzir a mesma divisão em execuções futuras.

In [3]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto,
                                         test_size=196,
                                         random_state=0)

Ajustamos uma regressão linear simples usando `horsepower` para prever `mpg`, mas apenas com as observações do conjunto de treino.

In [5]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

Em seguida, transformamos o conjunto de validação com a mesma especificação de modelo e calculamos o erro quadrático médio de validação. Esse valor estima como o modelo se comporta fora da amostra de treino.

Saída esperada: aproximadamente `23.6166`.

In [6]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((y_valid - valid_pred)**2)

np.float64(23.61661706966988)

Para evitar repetir código, criamos uma função `evalMSE()`. Ela recebe os termos do modelo, a variável resposta, o conjunto de treino e o conjunto de teste. Depois ajusta o modelo no treino e devolve o MSE no teste.

In [7]:
def evalMSE(terms,
            response,
            train,
            test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]
    X_test = mm.transform(test)
    y_test = test[response]
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean((y_test - test_pred)**2)

Agora comparamos regressões polinomiais de grau 1, 2 e 3. O objetivo é verificar se uma relação não linear entre `horsepower` e `mpg` reduz o erro de validação.

Saída esperada: aproximadamente `array([23.62, 18.76, 18.80])`.

In [11]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

Repetimos a divisão treino/validação com outra semente. A comparação mostra que a estimativa do erro por validação simples depende da divisão escolhida.

Saída esperada: aproximadamente `array([20.76, 16.95, 16.97])`.

In [12]:
Auto_train, Auto_valid = train_test_split(Auto,
                                         test_size=196,
                                         random_state=3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

Os resultados sustentam a interpretação já vista no capítulo: o modelo quadrático melhora bastante em relação ao linear, mas o cúbico não traz ganho claro.

## 5.3.2 Validação cruzada

Na teoria, a validação cruzada pode ser aplicada a qualquer modelo linear generalizado. Na prática, `sklearn` tem uma interface diferente de `statsmodels`. Para compatibilizar essas interfaces, o pacote `ISLP` usa o wrapper `sklearn_sm()`.

Esse wrapper permite passar modelos ajustados por `statsmodels` para ferramentas de validação cruzada do `sklearn`, como `cross_validate()`.

Aqui calculamos o erro por LOOCV, isto é, leave-one-out cross-validation. Como `cv=Auto.shape[0]`, cada observação é deixada fora uma vez, o modelo é ajustado nas demais observações e testado na observação deixada de fora.

Saída esperada: aproximadamente `24.2315`.

In [13]:
hp_model = sklearn_sm(sm.OLS,
                      MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,
                            X,
                            Y,
                            cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(24.231513517929212)

Repetimos o LOOCV para polinômios de grau 1 a 5. A queda forte do MSE entre grau 1 e grau 2 indica ganho relevante da curvatura. Depois disso, não aparece melhora sistemática.

Saída esperada: aproximadamente `array([24.2315, 19.2482, 19.3350, 19.4244, 19.0332])`.

In [14]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.42443031, 19.03320428])

A operação `outer()` aplica uma função a todos os pares de elementos entre dois vetores. Aqui, o exemplo usa soma. No bloco anterior, `np.power.outer()` foi usado para gerar colunas polinomiais.

Saída esperada: `array([[ 5, 7], [ 7, 9], [11, 13]])`.

In [15]:
A = np.array([3, 5, 9])
B = np.array([2, 4])
np.add.outer(A, B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

Agora usamos validação cruzada com 10 folds. Esse procedimento é mais rápido que LOOCV no uso genérico de `cross_validate()`, porque ajusta apenas 10 modelos por grau, não 392 modelos por grau.

Saída esperada: aproximadamente `array([24.2077, 19.1853, 19.2763, 19.4785, 19.1372])`.

In [16]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0) # use same splits for each degree
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848404, 19.13722016])

A função `cross_validate()` também aceita outros mecanismos de divisão. Com `ShuffleSplit()`, reproduzimos a lógica de um conjunto de validação aleatório.

Saída esperada: aproximadamente `array([23.6166])`.

In [17]:
validation = ShuffleSplit(n_splits=1,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation);
results['test_score']

array([23.61661707])

Se repetirmos várias divisões aleatórias, obtemos uma noção da variação de Monte Carlo associada à escolha dos conjuntos de treino e teste. A dispersão abaixo não é um erro padrão formal, pois os conjuntos de treino se sobrepõem.

Saída esperada: aproximadamente `(23.8022, 1.4218)`.

In [18]:
validation = ShuffleSplit(n_splits=10,
                          test_size=196,
                          random_state=0)
results = cross_validate(hp_model,
                         Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation)
results['test_score'].mean(), results['test_score'].std()

(np.float64(23.802232661034164), np.float64(1.4218450941091847))

## 5.3.3 Bootstrap

O bootstrap estima a variabilidade de uma estatística reamostrando observações com reposição. A ideia é construir muitas amostras artificiais do mesmo tamanho da amostra original e recalcular a estatística em cada uma delas.

### Estimando a acurácia de uma estatística de interesse

Usamos o conjunto `Portfolio`. A função `alpha_func()` calcula a estimativa de α a partir das observações indicadas pelo vetor `idx`. Esse α vem da fórmula de variância mínima apresentada no capítulo.

In [31]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X','Y']].iloc[idx], rowvar=False)
    return ((cov_[1,1] - cov_[0,1]) /
            (cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

Primeiro calculamos α usando todas as 100 observações originais.

Saída esperada: aproximadamente `0.5758`.

In [32]:
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

Agora selecionamos 100 observações com reposição. Isso equivale a criar uma amostra bootstrap e recalcular α nessa nova amostra.

Saída esperada: aproximadamente `0.6074`.

In [33]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
           rng.choice(100,
                      100,
                      replace=True))

np.float64(0.6074452469619004)

A função `boot_SE()` generaliza o processo. Ela recebe uma função, um dataframe, o tamanho da amostra bootstrap, o número de replicações `B` e uma semente aleatória. Em cada replicação, sorteia índices com reposição, calcula a estatística e acumula os termos necessários para estimar seu erro padrão.

O uso de `_` no laço indica que o contador em si não é usado, apenas queremos repetir o bloco `B` vezes.

In [40]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]

    for _ in range(B):
        idx = rng.choice(D.shape[0],
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2

    return np.sqrt(second_ / B - (first_ / B)**2)

Usamos 1.000 replicações bootstrap para estimar o erro padrão de α.

Saída esperada: aproximadamente `0.0912`.

In [41]:
alpha_SE = boot_SE(alpha_func,
                   Portfolio,
                   B=1000,
                   seed=0)
alpha_SE

np.float64(0.09118176521277699)

### Estimando a acurácia de um modelo de regressão linear

O bootstrap também pode avaliar a variabilidade de coeficientes de modelos estatísticos. Agora estimamos a variabilidade de β₀ e β₁ em uma regressão linear que usa `horsepower` para prever `mpg` no conjunto `Auto`.

Criamos `boot_OLS()`, uma função genérica para ajustar regressões em amostras bootstrap. A função `clone()` copia a especificação do modelo para que ela seja reajustada corretamente em cada reamostragem.

In [42]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.iloc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

A função `partial()` congela os primeiros argumentos de `boot_OLS()`. Assim, `hp_func` fica com a assinatura esperada por `boot_SE()`: recebe apenas o dataframe e os índices.

In [43]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

Abaixo geramos 10 amostras bootstrap e calculamos, em cada uma, o intercepto e o coeficiente de `horsepower`.

In [44]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

rng = np.random.default_rng(0)

np.array([hp_func(Auto,
                  rng.choice(392,
                             392,
                             replace=True)) for _ in range(10)])

array([[39.88064456, -0.1567849 ],
       [38.73298691, -0.14699495],
       [38.31734657, -0.14442683],
       [39.91446826, -0.15782234],
       [39.43349349, -0.15072702],
       [40.36629857, -0.15912217],
       [39.62334517, -0.15449117],
       [39.0580588 , -0.14952908],
       [38.66688437, -0.14521037],
       [39.64280792, -0.15555698]])

Agora estimamos os erros padrão bootstrap dos coeficientes com 1.000 replicações.

Saída esperada: aproximadamente `0.8488` para o intercepto e `0.0074` para `horsepower`.

In [45]:
hp_se = boot_SE(hp_func,
                Auto,
                B=1000,
                seed=10)
hp_se

intercept     0.848807
horsepower    0.007352
dtype: float64

Comparamos esses erros padrão bootstrap aos erros padrão tradicionais da regressão linear estimados por `statsmodels`.

Saída esperada: aproximadamente `0.717` para o intercepto e `0.006` para `horsepower`.

In [46]:
hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

A diferença entre os dois métodos não significa falha do bootstrap. O erro padrão clássico depende de suposições do modelo linear, enquanto o bootstrap não exige a mesma estrutura paramétrica. Como a relação entre `horsepower` e `mpg` é não linear, os resíduos do modelo linear simples podem inflar a estimativa de variância.

Por fim, repetimos a comparação com um modelo quadrático. Como o ajuste quadrático descreve melhor a relação entre `horsepower` e `mpg`, os erros padrão bootstrap e os erros padrão clássicos ficam mais próximos.

In [47]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS,
                    quad_model,
                    'mpg')
boot_SE(quad_func, Auto, B=1000)

intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64

Agora calculamos os erros padrão tradicionais do mesmo modelo quadrático com `sm.OLS()`.

In [48]:
M = sm.OLS(Auto['mpg'],
           quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64

## Fechamento

O laboratório mostra três usos complementares de reamostragem. A validação simples é intuitiva, mas sensível à divisão treino/teste. A validação cruzada usa melhor os dados e tende a dar uma estimativa mais estável do erro de teste. O bootstrap muda o foco: em vez de estimar erro preditivo, estima a variabilidade de estatísticas e coeficientes sem depender tanto de fórmulas paramétricas fechadas.